<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Phenology_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
admin_boundaries_fc = (
    ee.FeatureCollection("WM/geoLab/geoBoundaries/600/ADM2")
    .filterBounds(roi)
)
admin_boundaries_geometry = admin_boundaries_fc.geometry().simplify(100)

map.addLayer(admin_boundaries_geometry, {}, "border")

In [ ]:
ndvi = ee.ImageCollection("NOAA/CDR/VIIRS/NDVI/V1").select('NDVI','QA').filterDate('2015','2025')
ndvi

In [ ]:
def cloud_mask(img):
  ndvi_img = img.select('NDVI').multiply(0.0001)
  qa_img = img.select('QA')
  cloud = qa_img.bitwiseAnd(1 << 1).neq(0)
  shadow = qa_img.bitwiseAnd(1 << 2).neq(0)
  mask = cloud.Or(shadow).Not()
  return ndvi_img.updateMask(mask).copyProperties(img, img.propertyNames())

In [ ]:
ndvi_mask = ndvi.map(cloud_mask)
ndvi_mask

In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(
    ndvi_mask,
    engine='ee',
    scale=1,
    crs='EPSG:4326',
    geometry = (roi)
)

In [ ]:
yearly = ds.resample(time = 'Y').mean('time')
yearly

In [ ]:
import geopandas as gpd

In [ ]:
vec = geemap.ee_to_gdf(admin_boundaries_fc)
vec

In [ ]:
vec.plot()

In [ ]:
vec.crs

In [ ]:
vec.geometry

In [ ]:
ds_rename = ds.rename({"lon": "x", "lat": "y"})

In [ ]:
!pip install netCDF4
import netCDF4

In [ ]:
import xarray as xr

In [ ]:
import numpy as np

In [ ]:
yearly.NDVI.plot(x = 'time', marker = 'o')

In [ ]:
!pip install dea_tools

In [ ]:
!pip install netCDF4
import netCDF4

In [ ]:
import dea_tools.temporal

In [ ]:
ds.to_netcdf("ndvi_yearly.nc")
ds

In [ ]:
phenology = dea_tools.temporal.xr_phenology(
    ds.NDVI,
    stats=['SOS', 'POS', 'EOS', 'Trough', 'vSOS', 'vPOS', 'vEOS', 'LOS', 'AOS', 'ROG', 'ROS'], method_sos='first', method_eos='last', verbose=True
)

In [ ]:
phenology

In [ ]:
yearly = ds.resample(time = 'Y').mean('time')
yearly


In [ ]:
print(f"SOS for the point: {phenology.SOS.item()}")